# QLoRA Fine-Tuning - IlyaGusev/saiga_yandexgpt_8b

В этом блокноте приведён код fine-tuning'а модели IlyaGusev/saiga_yandexgpt_8b с помощью QLoRA на датасете из диалогов с консультантом.

Используется библиотека PEFT.

## Libraries

In [1]:
!pip install -U bitsandbytes
!pip install datasets
!pip install peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 12.6 MB/s eta 0:00:00


In [2]:
import json
import torch
from datasets import Dataset
from google.colab import userdata
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    GenerationConfig,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    BitsAndBytesConfig
)
from peft import get_peft_model, LoraConfig

## Dataset load

In [3]:
with open("/../data/processed/cleaned_dataset_json.json", "r", encoding="UTF-8") as f:
    dataset_json = json.load(f)

Промпт для модели

In [5]:
system_prompt = """
Твоя роль - это эксперт-консультант по подбору товаров для хобби.

Твоя задача - найти идеальные товары на основе истории диалога или уточнить предпочтения.

Входные данные:

История диалога: История взаимодействий.
Список товаров: Название, описание, ссылка.
Количество сообщений от меня: Число моих сообщений в истории диалога.

Выходные данные:
Если (Количество сообщений от меня < 3): Сгенерировать один релевантный и наводящий вопрос для уточнения хобби и предпочтений. Вопрос должен быть конкретным и направленным на получение полезной информации для подбора. Вопрос должен быть построен так, чтобы он был максимально полезен для определения стиля и предпочтений собеседника (например, для украшения подарка это может быть вопрос о том, в каком стиле должен быть украшен подарок, какие цвета предпочитает получатель и т.п.)

Иначе, если (Недостаточно информации для определения конкретных интересов ИЛИ Последнее сообщение от собеседника слишком общее и требует уточнения): Сгенерировать один релевантный и наводящий вопрос для уточнения хобби и предпочтений. Вопрос должен быть конкретным и направленным на получение полезной информации для подбора. Вопрос должен быть построен так, чтобы он был максимально полезен для определения стиля и предпочтений собеседника (например, для украшения подарка это может быть вопрос о том, в каком стиле должен быть украшен подарок, какие цвета предпочитает получатель и т.п.)

Иначе, если (Есть подходящие товары в списке):

Если несколько товаров, то выбрать 2 наиболее релевантных, используя конкретные названия, ссылки и описания:

Я думаю, вам могут подойти эти товары:
[Название товара 1]: [Описание товара 1] ([Ссылка на товар 1])
[Название товара 2]: [Описание товара 2] ([Ссылка на товар 2])
...
Если один товар, используя конкретные названия, ссылки и описания:

Мне кажется, вам отлично подойдет [Название товара]: [Описание товара] ([Ссылка на товар])

Если (Количество сообщений от меня > 5):

Выбрать наиболее подходящий товар в списке, используя конкретные названия, ссылки и описания:

Мне кажется, вам отлично подойдет [Название товара]: [Описание товара] ([Ссылка на товар])

Правила:

Действовать строго по алгоритму: Сначала проверить количество сообщений и достаточность информации. Затем искать подходящие товары.
Избегать лишних фраз и объяснений.
Быть конкретным: Использовать информацию из списка товаров и истории диалога.
Сохранять вежливый и полезный тон.
Быть максимально кратким.
Избегать слова ‘пользователь’.
При выводе товаров указывать конкретные названия, описания и ссылки.
При выводе товаров обязательно писать ссылку на этот товар.
При генерации вопроса учитывать, что последнее сообщение собеседника могло быть очень общим и требовать уточнения.
Важно: Сначала проверять количество сообщений. Если их меньше 3, независимо от содержания последнего сообщения, генерировать только вопросы.
Не генерировать ответ больше, чем на 100 слов.
Не задавай больше, чем 2 вопроса за раз.
Если количество сообщений от меня больше, чем 5, то независимо от содержания диалога выбирать товар из списка.
Выводи товары только из входного списка.
"""

In [6]:
texts = []

for dialog in dataset_json:
    messages = []

    for convers in dialog:
        messages.append({"role": "user", "content": convers["question"]})
        messages.append({"role": "assistant", "content": convers["answer"]})

    messages.pop()
    messages.append({"role": "system", "content": dialog[-1]["rag_content"] + "\n" + system_prompt})
    messages.append({"role": "assistant", "content": dialog[-1]["answer"]})

    texts.append(messages)

Пример диалога с предметами из RAG

In [7]:
dataset_json[0]

[{'question': 'Здравствуйте! Я хочу попробовать рисовать портреты, но пока не знаю, какие карандаши и бумагу лучше взять для начала.',
  'rag_content': "Выбранные товары:\nНазвание товара: Карандаши 6 цветов 'Принцесса', деревянные, шестигранные. Ссылка на товар: https://www.sima-land.ru/712034/karandashi-6-cvetov-princessa-derevyannye-shestigrannye/\nНазвание товара: Альбом для рисования А5, 40 листов на скрепке 'Карандаши', обложка мелованный картон, блок 100 г/м². Ссылка на товар: https://www.sima-land.ru/1246663/albom-dlya-risovaniya-a5-40-listov-na-skrepke-karandashi-oblozhka-melovannyy-karton-blok-100-g-m/\nНазвание товара: Карандаши художественные набор 12 штук (2H, H, HB, B, 2B, 3B, 4B, 5B, 6B, 8B, 10B, 12B). Ссылка на товар: https://www.sima-land.ru/7086102/karandashi-hudozhestvennye-nabor-12-shtuk-2h-h-hb-b-2b-3b-4b-5b-6b-8b-10b-12b/\nНазвание товара: Карандаши 12 цветов в тубусе, шестигранные, пластиковые. Ссылка на товар: https://www.sima-land.ru/7333128/karandashi-12-cveto

Tokenizer

In [8]:
MODEL_NAME = "IlyaGusev/saiga_yandexgpt_8b"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/18.0M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

Длина в токенах промпта и самого длинного диалога.

In [9]:
print(len(tokenizer(system_prompt)['input_ids']))

print(max(
    [len(tokenizer.apply_chat_template(x, tokenize=True, add_generation_prompt=True)) for x in texts]
))

643
2655


Датасет для обучения

In [10]:
def tokenize(examples):
    return tokenizer(examples["formatted_chat"], padding="max_length", truncation=True)


dataset = Dataset.from_dict({"chat": texts})

# formatted to model chat template
dataset = dataset.map(lambda x: {"formatted_chat": tokenizer.apply_chat_template(x["chat"], tokenize=False, add_generation_prompt=False)})

# tokenized
dataset_tokenized = dataset.map(tokenize, batched=True).remove_columns(['chat', 'formatted_chat'])

# train-test split
dataset_tokenized_split = dataset_tokenized.train_test_split(test_size=0.25)

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

## Model Fine-Tuning

In [11]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    llm_int8_enable_fp32_cpu_offload=True
)

model_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/276 [00:00<?, ?B/s]

In [12]:
peft_config = LoraConfig(
    r=4,
    lora_alpha=8,
    target_modules=[
        "k_proj",           # better connections between several parts of input
        "v_proj"            # better view of specific information
    ],
    lora_dropout=0.1,
    bias="none",
)
model = get_peft_model(model_base, peft_config)

model.print_trainable_parameters()

trainable params: 1,310,720 || all params: 8,037,863,424 || trainable%: 0.0163


In [15]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding="longest")

training_args = TrainingArguments(
    output_dir="mydir",
    learning_rate=1e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,
    optim="adamw_torch_4bit",
    dataloader_pin_memory=True,
    dataloader_num_workers=2,
    torch_empty_cache_steps=1
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_tokenized_split["train"],
    eval_dataset=dataset_tokenized_split["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
)

In [ ]:
trainer.train()

In [ ]:
model.save_pretrained("saiga-8b-ft-qlora")

In [ ]:
# small example

# generation_config = GenerationConfig.from_pretrained(MODEL_NAME)
# model.eval()

# rag_information = """
# Выбранные товары:\nНазвание товара: Карандаши 6 цветов 'Принцесса', деревянные, шестигранные. Ссылка на товар: https://www.sima-land.ru/712034/karandashi-6-cvetov-princessa-derevyannye-shestigrannye/\nНазвание товара: Альбом для рисования А5, 40 листов на скрепке 'Карандаши', обложка мелованный картон, блок 100 г/м². Ссылка на товар: https://www.sima-land.ru/1246663/albom-dlya-risovaniya-a5-40-listov-na-skrepke-karandashi-oblozhka-melovannyy-karton-blok-100-g-m/\nНазвание товара: Карандаши художественные набор 12 штук (2H, H, HB, B, 2B, 3B, 4B, 5B, 6B, 8B, 10B, 12B). Ссылка на товар: https://www.sima-land.ru/7086102/karandashi-hudozhestvennye-nabor-12-shtuk-2h-h-hb-b-2b-3b-4b-5b-6b-8b-10b-12b/\nНазвание товара: Карандаши 12 цветов в тубусе, шестигранные, пластиковые. Ссылка на товар: https://www.sima-land.ru/7333128/karandashi-12-cvetov-v-tubuse-shestigrannye-plastikovye/\nНазвание товара: Набор для рисования «Учимся рисовать: зверушки», с карандашами, с трафаретами, с декором, 3+. Ссылка на товар: https://www.sima-land.ru/5160956/nabor-dlya-risovaniya-uchimsya-risovat-zverushki-s-karandashami-s-trafaretami-s-dekorom-3-plus/\nКоличество сообщений от тебя: 0\n
# """
# s = [
#         {"role": "user", "content": "Здравствуйте! Я хочу попробовать рисовать портреты, но пока не знаю, какие карандаши и бумагу лучше взять для начала."},
#         {"role": "system", "content": rag_information + "\n" + system_prompt}
# ]
# prompt = tokenizer.apply_chat_template(s, tokenize=False, add_generation_prompt=True)

# data = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
# data = {k: v.to(model.device) for k, v in data.items()}
# data.pop("token_type_ids", None)

# output_ids = model.generate(**data, generation_config=generation_config, max_new_tokens=400)[0]
# output_ids = output_ids[len(data["input_ids"][0]):]

# output = tokenizer.decode(output_ids, skip_special_tokens=True).strip()

# torch.cuda.empty_cache()